# CNN for Natural Language processing- Demonstration In Python


## Background
With the rapid growth of online platforms, a huge amount of textual data is generated in the form of reviews, comments, and feedback. Understanding the sentiment behind this text (positive or negative) is crucial for businesses to make data-driven decisions.

Traditional Natural Language Processing (NLP) techniques rely on manual feature extraction, which can be time-consuming and less effective. Deep learning models, especially Convolutional Neural Networks (CNNs), have shown strong performance in automatically capturing important patterns and phrases in text data.

Here, we use the IMDB movie reviews dataset to build a sentiment analysis model using CNN.

## Objective:
To build a CNN-based model that classifies IMDB movie reviews as positive or negative.

### Import required libraries

In [6]:
from tensorflow.keras.layers import Input
import numpy as np
import tensorflow as tf

In [7]:
# Hugging Face dataset loader
from datasets import load_dataset

# Keras model and layers
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Embedding,
    Conv1D,
    GlobalMaxPooling1D,
    Dense
)

# Text preprocessing tools
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences


### Load IMDB Dataset

In [8]:
print("Loading IMDB dataset...")

# Load official IMDB dataset
dataset = load_dataset("imdb")

# Use small subset for faster training
# Shuffle dataset to ensure random and unbiased sampling
# seed=42 ensures reproducibility (same random order every run)
# Select a smaller subset (2000 samples) for faster training and experimentation
train_data = dataset["train"].shuffle(seed=42).select(range(2000))


# Apply same logic to test data
# Shuffle → avoids ordered bias
# Select 500 samples → speeds up evaluation while still being representative
test_data  = dataset["test"].shuffle(seed=42).select(range(500))

# Reviews (text)
X_train_text = train_data["text"]
X_test_text  = test_data["text"]

# Labels
# 0 = Negative review
# 1 = Positive review
y_train = np.array(train_data["label"])
y_test  = np.array(test_data["label"])

Loading IMDB dataset...


### Text Preprocessing

In [9]:
# Keep only top 5000 frequent words
vocab_size = 5000

# Every review will have 150 words
max_length = 150

# Tokenizer converts words into numbers
tokenizer = Tokenizer(
    num_words=vocab_size,
    oov_token="<OOV>"   # For unknown words
)

# Learn vocabulary from training reviews
tokenizer.fit_on_texts(X_train_text)

# Convert reviews into integer sequences
# Example:
# "movie was good" -> [45, 12, 98]
X_train_seq = tokenizer.texts_to_sequences(X_train_text)
X_test_seq  = tokenizer.texts_to_sequences(X_test_text)

# Make all reviews same length
# Short reviews -> add zeros
# Long reviews -> cut extra words
X_train_padded = pad_sequences(
    X_train_seq,
    maxlen=max_length,
    padding='post',
    truncating='post'
)

X_test_padded = pad_sequences(
    X_test_seq,
    maxlen=max_length,
    padding='post',
    truncating='post'
)


### Build CNN Model

In [10]:
model = Sequential([

    # ---------------------------------
    # INPUT LAYER
    # ---------------------------------
    # Defines the shape of input data
    # Each review is a sequence of 'max_length' words (integers)
    Input(shape=(max_length,)),

    
    # ---------------------------------
    # EMBEDDING LAYER
    # ---------------------------------
    # Converts each word (integer ID) into a dense vector
    # Think of it as learning word meanings
    # Output shape: (batch_size, max_length, 32)
    Embedding(
        input_dim=vocab_size,   # Total number of unique words
        output_dim=32           # Size of each word vector
    ),

    
    # ---------------------------------
    # CONVOLUTIONAL LAYER (CNN)
    # ---------------------------------
    # Detects important patterns in text
    # Example: phrases like "very good", "not bad"
    # kernel_size=5 → looks at 5-word groups at a time
    # filters=64 → learns 64 different types of patterns
    Conv1D(
        filters=64,
        kernel_size=5,
        activation='relu'
    ),


    # ---------------------------------
    # GLOBAL MAX POOLING
    # ---------------------------------
    # Picks the most important feature from each filter
    # Converts sequence into a fixed-size vector
    # Reduces data size and keeps strongest signals
    GlobalMaxPooling1D(),


    # ---------------------------------
    # FULLY CONNECTED (DENSE) LAYER
    # ---------------------------------
    # Learns higher-level combinations of features
    # Acts like a decision-making layer
    Dense(
        32,
        activation='relu'
    ),


    # ---------------------------------
    # OUTPUT LAYER
    # ---------------------------------
    # Produces final prediction
    # Sigmoid → outputs probability (0 to 1)
    # 0 → Negative review
    # 1 → Positive review
    Dense(
        1,
        activation='sigmoid'
    )

])

### Compile Model

In [11]:
model.compile(

    # Optimizer updates weights
    optimizer='adam',

    # Binary classification loss
    loss='binary_crossentropy',

    # Accuracy metric
    metrics=['accuracy']
)


# Show model structure
model.summary()


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)              │ (None, 150, 32)             │         160,000 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv1d_2 (Conv1D)                    │ (None, 146, 64)             │          10,304 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ global_max_pooling1d_2               │ (None, 64)                  │               0 │
│ (GlobalMaxPooling1D)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_4 (Dense)                      │ (None, 32)                  │           2,080 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_5 (Dense)                      │ (None, 1)                   │              33 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 172,417 (673.50 KB)

 Trainable params: 172,417 (673.50 KB)

 Non-trainable params: 0 (0.00 B)

### Train Model

In [12]:
print("\nTraining CNN model...\n")

model.fit(

    # Training data
    X_train_padded,
    y_train,

    # Number of full passes
    epochs=5,

    # Reviews processed together
    batch_size=32,

    # Validation data
    validation_data=(X_test_padded, y_test)
)



Training CNN model...

Epoch 1/5
63/63 ━━━━━━━━━━━━━━━━━━━━ 5s 31ms/step - accuracy: 0.4955 - loss: 0.6938 - val_accuracy: 0.5100 - val_loss: 0.6910
Epoch 2/5
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - accuracy: 0.7500 - loss: 0.6735 - val_accuracy: 0.6720 - val_loss: 0.6363
Epoch 3/5
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - accuracy: 0.7595 - loss: 0.5416 - val_accuracy: 0.7500 - val_loss: 0.5206
Epoch 4/5
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.8865 - loss: 0.3291 - val_accuracy: 0.7800 - val_loss: 0.4645
Epoch 5/5
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.9780 - loss: 0.1331 - val_accuracy: 0.7700 - val_loss: 0.4836


### Evaluate Model

In [13]:
loss, accuracy = model.evaluate(
    X_test_padded,
    y_test,
    verbose=0
)

print(f"\nFinal Test Accuracy: {accuracy * 100:.2f}%")


Final Test Accuracy: 77.00%
